In [1]:
import sys; print(sys.executable)

C:\Projects\telecom-vendor-rebate-analytics\.venv\Scripts\python.exe


In [2]:
from pathlib import Path
import pandas as pd

RAW = Path(r"C:\Projects\telecom-vendor-rebate-analytics\data\raw")

TABLES = [
    "dim_vendor", "dim_date", "business_rules",
    "installation_transaction", "po_header", "po_line",
    "invoice_header", "invoice_line", "payment",
    "payment_allocation", "workflow_event", "defect_injection_log",
]

# Read everything as text, and keep blanks as empty strings (not NaN)
data = {
    name: pd.read_csv(RAW / f"{name}.csv", dtype=str, keep_default_na=False)
    for name in TABLES
}

summary = pd.DataFrame({
    "table": TABLES,
    "rows": [len(data[n]) for n in TABLES],
    "columns": [data[n].shape[1] for n in TABLES],
})
print(summary.to_string(index=False))

                   table  rows  columns
              dim_vendor    35        6
                dim_date   730       10
          business_rules    11        6
installation_transaction  6030       11
               po_header  1256        5
                 po_line  5322        5
          invoice_header  1772        9
            invoice_line  5788        5
                 payment  1581        6
      payment_allocation  1662        5
          workflow_event 21051        8
    defect_injection_log  1275        8


In [3]:
MANDATORY = {
    "installation_transaction": ["transaction_id", "vendor_id", "project_code", "installation_date", "property_type"],
    "po_header": ["po_id", "vendor_id", "po_date"],
    "invoice_header": ["invoice_id", "vendor_id", "invoice_reference", "invoice_date", "po_id"],
    "payment": ["payment_id", "vendor_id", "payment_reference", "payment_date"],
    "workflow_event": ["event_id", "to_status", "event_timestamp"],
}

rows = []
for table, cols in MANDATORY.items():
    for col in cols:
        blanks = int((data[table][col].str.strip() == "").sum())
        rows.append({"table": table, "column": col, "blank_count": blanks})

missing = pd.DataFrame(rows)
print(missing.to_string(index=False))

                   table            column  blank_count
installation_transaction    transaction_id            0
installation_transaction         vendor_id           24
installation_transaction      project_code           24
installation_transaction installation_date           18
installation_transaction     property_type            0
               po_header             po_id            0
               po_header         vendor_id            0
               po_header           po_date            0
          invoice_header        invoice_id            0
          invoice_header         vendor_id            0
          invoice_header invoice_reference            0
          invoice_header      invoice_date            5
          invoice_header             po_id           12
                 payment        payment_id            0
                 payment         vendor_id            0
                 payment payment_reference            0
                 payment      payment_date      

In [4]:
def duplicate_groups(df, keys):
    """Return the rows whose key combination appears more than once."""
    return df[df.duplicated(subset=keys, keep=False)]

inst = data["installation_transaction"]
pol = data["po_line"]
inv = data["invoice_header"]
pay = data["payment"]

checks = {
    "Installations sharing a property_ref": duplicate_groups(inst, ["property_ref"]),
    "PO lines sharing a transaction_id": duplicate_groups(pol, ["transaction_id"]),
    "Invoices sharing vendor + invoice_reference": duplicate_groups(inv, ["vendor_id", "invoice_reference"]),
    "Payments sharing vendor + payment_reference": duplicate_groups(pay, ["vendor_id", "payment_reference"]),
}

for label, df in checks.items():
    print(f"{label}: {len(df)} rows involved")

Installations sharing a property_ref: 60 rows involved
PO lines sharing a transaction_id: 52 rows involved
Invoices sharing vendor + invoice_reference: 38 rows involved
Payments sharing vendor + payment_reference: 18 rows involved


In [5]:
REPORTING_DATE = pd.Timestamp("2026-09-30")

# 1. Invalid property type (exact match, so "FLAT " and "house" fail)
bad_type = inst[~inst["property_type"].isin(["Flat", "House"])]

# 2. Installation date after the reporting date (ignore blanks here)
dates = pd.to_datetime(inst["installation_date"], errors="coerce")
future = inst[dates > REPORTING_DATE]

# 3. Negative invoice line values
inl = data["invoice_line"]
neg_inl = inl[pd.to_numeric(inl["invoice_line_value"]) < 0]

# 4. Zero or negative allocation amounts
alc = data["payment_allocation"]
bad_alc = alc[pd.to_numeric(alc["allocation_amount"]) <= 0]

print("Invalid property type:        ", len(bad_type))
print("Installation date in future:  ", len(future))
print("Negative invoice line value:  ", len(neg_inl))
print("Zero/negative allocation:     ", len(bad_alc))
print()
print("Property type values found:")
print(inst["property_type"].value_counts().to_string())

Invalid property type:         24
Installation date in future:   18
Negative invoice line value:   17
Zero/negative allocation:      5

Property type values found:
property_type
Flat        3620
House       2386
Unknown        7
Bungalow       6
Flt            5
FLAT           3
house          3


In [6]:
def orphans(child, child_col, parent, parent_col, ignore_blank=True):
    """Count child rows whose link value is not found in the parent table."""
    values = child[child_col]
    if ignore_blank:
        values = values[values.str.strip() != ""]
    return int((~values.isin(parent[parent_col])).sum())

vendor = data["dim_vendor"]
po_h = data["po_header"]
pol = data["po_line"]
inv = data["invoice_header"]
inl = data["invoice_line"]
alc = data["payment_allocation"]
pay = data["payment"]
inst = data["installation_transaction"]

links = [
    ("installation_transaction.vendor_id -> dim_vendor", orphans(inst, "vendor_id", vendor, "vendor_id")),
    ("po_header.vendor_id -> dim_vendor", orphans(po_h, "vendor_id", vendor, "vendor_id")),
    ("po_line.po_id -> po_header", orphans(pol, "po_id", po_h, "po_id")),
    ("po_line.transaction_id -> installation_transaction", orphans(pol, "transaction_id", inst, "transaction_id")),
    ("invoice_header.po_id -> po_header (blanks ignored)", orphans(inv, "po_id", po_h, "po_id")),
    ("invoice_line.invoice_id -> invoice_header", orphans(inl, "invoice_id", inv, "invoice_id")),
    ("invoice_line.po_line_id -> po_line", orphans(inl, "po_line_id", pol, "po_line_id")),
    ("payment_allocation.payment_id -> payment", orphans(alc, "payment_id", pay, "payment_id")),
    ("payment_allocation.invoice_id -> invoice_header", orphans(alc, "invoice_id", inv, "invoice_id")),
]

print(pd.DataFrame(links, columns=["link", "orphan_rows"]).to_string(index=False))

                                              link  orphan_rows
  installation_transaction.vendor_id -> dim_vendor            0
                 po_header.vendor_id -> dim_vendor            0
                        po_line.po_id -> po_header            0
po_line.transaction_id -> installation_transaction            0
invoice_header.po_id -> po_header (blanks ignored)            0
         invoice_line.invoice_id -> invoice_header            0
                invoice_line.po_line_id -> po_line           13
          payment_allocation.payment_id -> payment            0
   payment_allocation.invoice_id -> invoice_header            8


In [7]:
RATE = {"Flat": 60.00, "House": 90.00}

i = data["installation_transaction"].copy()
i["recorded"] = pd.to_numeric(i["expected_rebate_recorded"])
i["is_eligible_n"] = pd.to_numeric(i["is_eligible"])

# Recalculate only where property type is valid
valid = i["property_type"].isin(RATE.keys())
i["recalculated"] = i["property_type"].map(RATE).where(i["is_eligible_n"] == 1, 0.0)

wrong = i[valid & (i["recorded"].round(2) != i["recalculated"].round(2))]
print("Installations with valid type:", int(valid.sum()))
print("Recorded rebate differs from recalculated:", len(wrong))
print()
print(wrong[["transaction_id", "property_type", "completion_status", "is_eligible", "recorded", "recalculated"]].head(8).to_string(index=False))

# PO line value vs the installation's recorded rebate
p = data["po_line"].copy()
p["po_value"] = pd.to_numeric(p["po_line_value"])
m = p.merge(i[["transaction_id", "recorded"]], on="transaction_id", how="left")
diff = m[(m["po_value"] - m["recorded"]).abs() > 0.004]
print()
print("PO lines whose value differs from the installation's recorded rebate:", len(diff))

Installations with valid type: 6006
Recorded rebate differs from recalculated: 120

transaction_id property_type completion_status is_eligible  recorded  recalculated
       T000009          Flat         Completed           1      45.0          60.0
       T000081         House         Completed           1      60.0          90.0
       T000095          Flat         Completed           1     120.0          60.0
       T000179          Flat         Completed           1      90.0          60.0
       T000285          Flat         Completed           1      70.0          60.0
       T000290          Flat         Completed           1      70.0          60.0
       T000348          Flat         Completed           1      45.0          60.0
       T000386         House         Completed           1      60.0          90.0

PO lines whose value differs from the installation's recorded rebate: 253


In [8]:
# Uses i (with 'recalculated' and 'valid') and p from the previous cell
ref = i[valid][["transaction_id", "recalculated"]].drop_duplicates("transaction_id")
m2 = p.merge(ref, on="transaction_id", how="inner")
diff2 = m2[(m2["po_value"] - m2["recalculated"]).abs() > 0.004]

print("PO lines compared with the recalculated rebate:", len(m2))
print("PO lines differing from the rule:", len(diff2))

# Separate eligibility check
bad_elig = i[(i["completion_status"] != "Completed") & (i["is_eligible_n"] == 1)]
print("Eligible but not Completed:", len(bad_elig))

PO lines compared with the recalculated rebate: 5299
PO lines differing from the rule: 148
Eligible but not Completed: 24


In [9]:
pay = data["payment"].copy()
alc = data["payment_allocation"].copy()
pay["payment_amount_n"] = pd.to_numeric(pay["payment_amount"])
alc["allocation_amount_n"] = pd.to_numeric(alc["allocation_amount"])

# 1. Allocations summed per payment vs the payment amount
alloc_sum = alc.groupby("payment_id")["allocation_amount_n"].sum().rename("alloc_total")
chk = pay.merge(alloc_sum, on="payment_id", how="left")
chk["alloc_total"] = chk["alloc_total"].fillna(0)
mismatch = chk[(chk["payment_amount_n"] - chk["alloc_total"]).abs() > 0.004]
print("Payments where allocations differ from payment amount:", len(mismatch))

# 2. Completed allocations summed per invoice vs invoice total
inl = data["invoice_line"].copy()
inl["value_n"] = pd.to_numeric(inl["invoice_line_value"])
inv_total = inl.groupby("invoice_id")["value_n"].sum().rename("invoice_total")

done = alc.merge(pay[["payment_id", "payment_status"]], on="payment_id")
done = done[done["payment_status"] == "Completed"]
paid = done.groupby("invoice_id")["allocation_amount_n"].sum().rename("paid_total")

both = pd.concat([inv_total, paid], axis=1).dropna()
over = both[both["paid_total"] - both["invoice_total"] > 0.004]
print("Invoices paid more than their total:", len(over))

Payments where allocations differ from payment amount: 21
Invoices paid more than their total: 49


In [10]:
neg_invoices = set(inl[inl["value_n"] < 0]["invoice_id"])
n_alloc = done.groupby("invoice_id").size()

over = over.copy()
over["excess"] = (over["paid_total"] - over["invoice_total"]).round(2)
over["has_negative_line"] = over.index.isin(neg_invoices)
over["completed_allocations"] = n_alloc.reindex(over.index)

print("Overpaid invoices:", len(over))
print("With a negative invoice line:", int(over["has_negative_line"].sum()))
print()
print("Completed allocations per overpaid invoice (negative-line invoices excluded):")
print(over[~over["has_negative_line"]]["completed_allocations"].value_counts().sort_index().to_string())

Overpaid invoices: 49
With a negative invoice line: 14

Completed allocations per overpaid invoice (negative-line invoices excluded):
completed_allocations
1    13
2    18
3     4


In [11]:
dl = data["defect_injection_log"]

def invoices_for(defect_types, lookup, id_col):
    """Invoices linked to logged defects, via the table that holds the record id."""
    ids = dl[dl["defect_type"].isin(defect_types)]["record_id"]
    return set(lookup[lookup[id_col].isin(ids)]["invoice_id"])

from_overpay = invoices_for(["Overpayment"], alc, "allocation_id")
from_dups = invoices_for(["Duplicate payment (same reference)", "Potential duplicate payment"], alc, "payment_id")
from_neg = invoices_for(["Negative invoice line value"], inl, "invoice_line_id")

def cause(inv_id):
    found = []
    if inv_id in from_neg: found.append("negative line")
    if inv_id in from_overpay: found.append("overpayment")
    if inv_id in from_dups: found.append("duplicate payment")
    return " + ".join(found) if found else "UNEXPLAINED"

over["cause"] = [cause(i) for i in over.index]
print(over["cause"].value_counts().to_string())

cause
duplicate payment    18
overpayment          17
negative line        14


In [12]:
# 1. Amount summaries (convert text to numbers first)
amounts = {
    "installation rebate": pd.to_numeric(data["installation_transaction"]["expected_rebate_recorded"]),
    "PO line value": pd.to_numeric(data["po_line"]["po_line_value"]),
    "invoice line value": pd.to_numeric(data["invoice_line"]["invoice_line_value"]),
    "payment amount": pd.to_numeric(data["payment"]["payment_amount"]),
    "allocation amount": pd.to_numeric(data["payment_allocation"]["allocation_amount"]),
}
summary = pd.DataFrame({k: v.describe() for k, v in amounts.items()}).T
print(summary[["count", "min", "50%", "mean", "max"]].round(2).to_string())

# 2. Which rebate values exist?
print()
print("Recorded rebate values:")
print(amounts["installation rebate"].value_counts().sort_index().to_string())

# 3. Installations per month (blank dates are skipped)
d = pd.to_datetime(data["installation_transaction"]["installation_date"], errors="coerce")
print()
print("Blank installation dates:", int(d.isna().sum()))
print("Installations per month:")
print(d.dt.to_period("M").value_counts().sort_index().to_string())

                      count    min    50%    mean     max
installation rebate  6030.0    0.0   60.0   64.86   120.0
PO line value        5322.0   30.0   60.0   72.39   150.0
invoice line value   5788.0  -90.0   60.0   72.19   190.0
payment amount       1581.0   12.0  180.0  230.41  1410.0
allocation amount    1662.0 -480.0  168.0  217.59   720.0

Recorded rebate values:
expected_rebate_recorded
0.0       600
10.0        2
45.0       31
60.0     3218
70.0       13
90.0     2132
100.0      14
120.0      20

Blank installation dates: 18
Installations per month:
installation_date
2025-10    554
2025-11    549
2025-12    557
2026-01    533
2026-02    469
2026-03    532
2026-04    538
2026-05    523
2026-06    559
2026-07    577
2026-08    603
2026-11      2
2026-12      7
2027-01      2
2027-02      5
2027-04      2
Freq: M


In [13]:
exceptions = [
    ("Missing vendor_id (installation)", int((inst["vendor_id"].str.strip() == "").sum())),
    ("Missing project_code (installation)", int((inst["project_code"].str.strip() == "").sum())),
    ("Missing installation_date", int((inst["installation_date"].str.strip() == "").sum())),
    ("Missing invoice_date", int((inv["invoice_date"].str.strip() == "").sum())),
    ("Invoice without PO number", int((inv["po_id"].str.strip() == "").sum())),
    ("Invalid property type", len(bad_type)),
    ("Installation date in future", len(future)),
    ("Wrong rebate (recorded vs rule)", len(wrong)),
    ("Eligible but not Completed", len(bad_elig)),
    ("PO line differs from rule", len(diff2)),
    ("Duplicate property_ref (extra copies)", int(inst.duplicated(subset=["property_ref"]).sum())),
    ("Installation on two PO lines (extra copies)", int(pol.duplicated(subset=["transaction_id"]).sum())),
    ("Duplicate invoice reference (extra copies)", int(inv.duplicated(subset=["vendor_id", "invoice_reference"]).sum())),
    ("Duplicate payment reference (extra copies)", int(pay.duplicated(subset=["vendor_id", "payment_reference"]).sum())),
    ("Negative invoice line value", len(neg_inl)),
    ("Zero or negative allocation", len(bad_alc)),
    ("Invoice line with no matching PO line", orphans(inl, "po_line_id", pol, "po_line_id")),
    ("Allocation to a non-existent invoice", orphans(alc, "invoice_id", inv, "invoice_id")),
    ("Payment differs from its allocations", len(mismatch)),
    ("Invoice paid more than its total", len(over)),
]

freq = pd.DataFrame(exceptions, columns=["check", "records_flagged"])
print(freq.to_string(index=False))

                                      check  records_flagged
           Missing vendor_id (installation)               24
        Missing project_code (installation)               24
                  Missing installation_date               18
                       Missing invoice_date                5
                  Invoice without PO number               12
                      Invalid property type               24
                Installation date in future               18
            Wrong rebate (recorded vs rule)              120
                 Eligible but not Completed               24
                  PO line differs from rule              148
      Duplicate property_ref (extra copies)               30
Installation on two PO lines (extra copies)               26
 Duplicate invoice reference (extra copies)               19
 Duplicate payment reference (extra copies)                9
                Negative invoice line value               17
                Zero or 

In [14]:
OUT = Path(r"C:\Projects\telecom-vendor-rebate-analytics\data\processed")
OUT.mkdir(parents=True, exist_ok=True)
freq.to_csv(OUT / "profiling_exception_summary.csv", index=False)
print("Saved:", OUT / "profiling_exception_summary.csv")

Saved: C:\Projects\telecom-vendor-rebate-analytics\data\processed\profiling_exception_summary.csv
